# Intermediate Analysis — Questions 11-20

**Techniques:** `CTE` · `RANK()` · `DENSE_RANK()` · `ROW_NUMBER()` · `NTILE()` · `LAG()` · `LEAD()` · cumulative `SUM OVER` · moving average `ROWS BETWEEN`

| # | Question |
|---|----------|
| Q11 | Monthly revenue trend — best and worst months |
| Q12 | Actual vs estimated delivery time gap |
| Q13 | Top 3 best-selling products per category |
| Q14 | Top 10% sellers — share of total revenue |
| Q15 | Weekday vs weekend order behavior |
| Q16 | First order per customer (deduplication) |
| Q17 | RANK vs DENSE_RANK — revenue ranking within category |
| Q18 | Cumulative revenue over time |
| Q19 | LEAD() — compare each month to the next |
| Q20 | 3-month moving average of order volume |

In [ ]:
import sys
sys.path.append('../')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.db_utils import run_query
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## Q11 — Monthly Revenue Trend
`CTE`, `RANK()`

In [ ]:
q11 = run_query("""
    WITH mr AS (
        SELECT STRFTIME('%Y-%m', o.order_purchase_timestamp) AS month,
               ROUND(SUM(op.payment_value), 2) AS revenue
        FROM orders o JOIN order_payments op ON o.order_id = op.order_id
        WHERE o.order_status = 'delivered' GROUP BY month
    )
    SELECT month, revenue, RANK() OVER (ORDER BY revenue DESC) AS rk FROM mr ORDER BY month
""")
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(q11)), q11['revenue'], marker='o', color='steelblue', linewidth=2)
ax.fill_between(range(len(q11)), q11['revenue'], alpha=0.15, color='steelblue')
best = q11[q11['rk'] == 1].iloc[0]
best_i = q11[q11['rk'] == 1].index[0]
ax.annotate(f"Best: {best['month']}\n${best['revenue']:,.0f}",
            xy=(best_i, best['revenue']),
            xytext=(best_i - 3, best['revenue'] * 0.88),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=9, color='green')
ax.set_xticks(range(len(q11)))
ax.set_xticklabels(q11['month'], rotation=45, ha='right', fontsize=8)
ax.set_title('Q11 — Monthly Revenue Trend (Delivered Orders)', fontweight='bold')
ax.set_ylabel('Revenue (BRL)')
plt.tight_layout(); plt.savefig('../images/q11_monthly_revenue.png', bbox_inches='tight'); plt.show()
q11.nlargest(5, 'revenue')[['month', 'revenue', 'rk']]

## Q12 — Actual vs Estimated Delivery Time
`JULIANDAY`, `CASE WHEN`

In [ ]:
q12 = run_query("""
    SELECT ROUND(AVG(JULIANDAY(order_delivered_customer_date) - JULIANDAY(order_purchase_timestamp)), 1) AS avg_actual_days,
           ROUND(AVG(JULIANDAY(order_estimated_delivery_date) - JULIANDAY(order_purchase_timestamp)), 1) AS avg_estimated_days,
           ROUND(AVG(JULIANDAY(order_estimated_delivery_date) - JULIANDAY(order_delivered_customer_date)), 1) AS avg_days_early,
           COUNT(*) AS total_orders
    FROM orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
""")
fig, ax = plt.subplots(figsize=(7, 5))
labels = ['Actual Delivery', 'Estimated Delivery']
values = [q12['avg_actual_days'].values[0], q12['avg_estimated_days'].values[0]]
bars = ax.bar(labels, values, color=['#2ecc71', '#3498db'], width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val}d', ha='center', fontweight='bold')
ax.set_title(f'Q12 — Delivery Time: Actual vs Estimated\nOrders arrive {q12["avg_days_early"].values[0]} days early on average', fontweight='bold')
ax.set_ylabel('Days')
plt.tight_layout(); plt.savefig('../images/q12_delivery_time.png', bbox_inches='tight'); plt.show()
q12

## Q13 — Top 3 Products per Category
`RANK()`, `PARTITION BY`

In [ ]:
q13 = run_query("""
    WITH rp AS (
        SELECT t.product_category_name_english AS category, oi.product_id,
               COUNT(oi.order_id) AS total_sold,
               RANK() OVER (PARTITION BY t.product_category_name_english ORDER BY COUNT(oi.order_id) DESC) AS rnk
        FROM order_items oi JOIN products p ON oi.product_id = p.product_id
        JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
        GROUP BY category, oi.product_id
    )
    SELECT * FROM rp WHERE rnk <= 3 ORDER BY category, rnk
""")
print(f'Categories analyzed: {q13["category"].nunique()}')
print(f'Total records (top 3 per category): {len(q13)}')
q13.head(15)

## Q14 — Top 10% Sellers: Revenue Concentration
`NTILE()`, `CTE`

In [ ]:
q14 = run_query("""
    WITH sr AS (
        SELECT seller_id, ROUND(SUM(price), 2) AS rev,
               NTILE(10) OVER (ORDER BY SUM(price) DESC) AS decile
        FROM order_items GROUP BY seller_id
    ), t AS (SELECT SUM(rev) AS grand FROM sr)
    SELECT CASE WHEN decile = 1 THEN 'Top 10%' ELSE 'Bottom 90%' END AS grp,
           COUNT(*) AS cnt, ROUND(SUM(rev), 2) AS rev,
           ROUND(SUM(rev) * 100.0 / MAX(grand), 2) AS pct
    FROM sr, t GROUP BY grp
""")
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(q14['rev'], labels=q14['grp'], autopct='%1.1f%%',
       colors=['#e74c3c', '#95a5a6'], startangle=90)
ax.set_title('Q14 — Revenue: Top 10% vs Bottom 90% Sellers', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q14_seller_concentration.png', bbox_inches='tight'); plt.show()
q14

## Q15 — Weekday vs Weekend Order Behavior
`STRFTIME`, window `COUNT`

In [ ]:
q15 = run_query("""
    SELECT CASE WHEN CAST(STRFTIME('%w', order_purchase_timestamp) AS INTEGER) IN (0,6)
                THEN 'Weekend' ELSE 'Weekday' END AS day_type,
           COUNT(*) AS total_orders,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
    FROM orders GROUP BY day_type
""")
fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(data=q15, x='day_type', y='total_orders', ax=ax,
            hue='day_type', palette=['#3498db', '#e67e22'], legend=False)
for i, row in q15.iterrows():
    ax.text(i, row['total_orders'] + 200, f"{row['pct']}%", ha='center', fontweight='bold')
ax.set_title('Q15 — Weekday vs Weekend Orders', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q15_weekday_weekend.png', bbox_inches='tight'); plt.show()
q15

## Q16 — First Order per Customer (Deduplication Pattern)
`ROW_NUMBER()` — most common interview pattern

In [ ]:
q16 = run_query("""
    WITH ranked_orders AS (
        SELECT c.customer_unique_id, o.order_id, o.order_purchase_timestamp,
               ROW_NUMBER() OVER (
                   PARTITION BY c.customer_unique_id
                   ORDER BY o.order_purchase_timestamp ASC
               ) AS rn
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
    )
    SELECT customer_unique_id, order_id, order_purchase_timestamp
    FROM ranked_orders WHERE rn = 1
    LIMIT 20
""")
print('ROW_NUMBER() assigns rank 1 to the FIRST order per customer.')
print(f'Result: {len(q16)} unique customers, each showing their earliest order.')
q16.head(10)

## Q17 — RANK vs DENSE_RANK: What is the Difference?
`RANK()` leaves gaps for ties · `DENSE_RANK()` does not

In [ ]:
q17 = run_query("""
    WITH cat_rev AS (
        SELECT t.product_category_name_english AS category, oi.product_id,
               ROUND(SUM(oi.price), 2) AS revenue
        FROM order_items oi JOIN products p ON oi.product_id = p.product_id
        JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
        GROUP BY category, oi.product_id
    )
    SELECT category, product_id, revenue,
           RANK()       OVER (PARTITION BY category ORDER BY revenue DESC) AS rank_gaps,
           DENSE_RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS dense_rank_no_gaps
    FROM cat_rev WHERE category = 'computers_accessories'
    ORDER BY revenue DESC LIMIT 20
""")
print('RANK vs DENSE_RANK demonstration on computers_accessories:')
print('When two products have the same revenue, RANK() skips the next number,')
print('DENSE_RANK() does not skip — it assigns consecutive ranks.')
q17

## Q18 — Cumulative Revenue Over Time
`SUM() OVER (ORDER BY)` — running total pattern

In [ ]:
q18 = run_query("""
    WITH monthly_rev AS (
        SELECT STRFTIME('%Y-%m', o.order_purchase_timestamp) AS month,
               ROUND(SUM(op.payment_value), 2) AS revenue
        FROM orders o JOIN order_payments op ON o.order_id = op.order_id
        WHERE o.order_status = 'delivered' GROUP BY month
    )
    SELECT month, revenue,
           ROUND(SUM(revenue) OVER (ORDER BY month), 2) AS cumulative_revenue
    FROM monthly_rev ORDER BY month
""")
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(q18)), q18['revenue'], color='steelblue', alpha=0.5, label='Monthly Revenue')
ax2 = ax.twinx()
ax2.plot(range(len(q18)), q18['cumulative_revenue'], color='#e74c3c', linewidth=2.5, marker='o', markersize=4, label='Cumulative')
ax.set_xticks(range(len(q18)))
ax.set_xticklabels(q18['month'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Monthly Revenue (BRL)', color='steelblue')
ax2.set_ylabel('Cumulative Revenue (BRL)', color='#e74c3c')
ax.set_title('Q18 — Monthly vs Cumulative Revenue', fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout(); plt.savefig('../images/q18_cumulative_revenue.png', bbox_inches='tight'); plt.show()
q18.tail(5)

## Q19 — LEAD(): Compare Each Month to the Next
`LEAD()` — looking forward (opposite of `LAG`)

In [ ]:
q19 = run_query("""
    WITH monthly_rev AS (
        SELECT STRFTIME('%Y-%m', o.order_purchase_timestamp) AS month,
               ROUND(SUM(op.payment_value), 2) AS revenue
        FROM orders o JOIN order_payments op ON o.order_id = op.order_id
        WHERE o.order_status = 'delivered' GROUP BY month
    )
    SELECT month, revenue,
           LEAD(revenue) OVER (ORDER BY month) AS next_month_revenue,
           ROUND((LEAD(revenue) OVER (ORDER BY month) - revenue) * 100.0 / NULLIF(revenue, 0), 2) AS expected_change_pct
    FROM monthly_rev ORDER BY month
""")
q19_clean = q19.dropna()
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in q19_clean['expected_change_pct']]
ax.bar(range(len(q19_clean)), q19_clean['expected_change_pct'], color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(len(q19_clean)))
ax.set_xticklabels(q19_clean['month'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Expected Change (%)')
ax.set_title('Q19 — Month-on-Month Expected Change (using LEAD)', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q19_lead_mom_change.png', bbox_inches='tight'); plt.show()
q19.head(10)

## Q20 — 3-Month Moving Average of Order Volume
`AVG() OVER (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)`

In [ ]:
q20 = run_query("""
    WITH monthly_orders AS (
        SELECT STRFTIME('%Y-%m', order_purchase_timestamp) AS month,
               COUNT(*) AS order_count
        FROM orders GROUP BY month
    )
    SELECT month, order_count,
           ROUND(AVG(order_count) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS moving_avg_3m
    FROM monthly_orders ORDER BY month
""")
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(q20)), q20['order_count'], color='steelblue', alpha=0.4, label='Monthly Orders')
ax.plot(range(len(q20)), q20['moving_avg_3m'], color='#e74c3c', linewidth=2.5, marker='o', markersize=4, label='3-Month Moving Avg')
ax.set_xticks(range(len(q20)))
ax.set_xticklabels(q20['month'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Order Count')
ax.set_title('Q20 — Monthly Orders with 3-Month Moving Average', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.savefig('../images/q20_moving_average.png', bbox_inches='tight'); plt.show()
q20.tail(10)